In [1]:
!pip install yfinance


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

In [3]:
df = yf.download("AAPL", period="5y", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

[*********************100%***********************]  1 of 1 completed


In [4]:

df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"] = df["Close"].pct_change()
df["return_3d"] = df["Close"].pct_change(3)
df["return_5d"] = df["Close"].pct_change(5)

df["volatility_5d"] = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

df["target"] = (df["return"].shift(-1) > 0.002).astype(int)
df = df.dropna()

In [5]:
feature_cols = [
    "rsi_14",
    "macd",
    "macd_signal",
    "bb_width",
    "return_3d",
    "return_5d",
    "volatility_5d",
    "volatility_10d",
    "dist_sma_10",
    "dist_sma_20",
    "Volume"
]

X = df[feature_cols]
y = df["target"]

split_index = int(len(df) * 0.8)

X_train = X[:split_index]
X_test  = X[split_index:]
y_train = y[:split_index]
y_test  = y[split_index:]

In [6]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [7]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=100
)

c:\Users\deera\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [14:16:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-logloss:0.68925
[100]	validation_0-logloss:0.69845
[200]	validation_0-logloss:0.71305
[300]	validation_0-logloss:0.72076
[400]	validation_0-logloss:0.74161
[499]	validation_0-logloss:0.75586


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [8]:
xgb_pred = xgb.predict(X_test_scaled)

print("XGB Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))

XGB Accuracy: 0.5241935483870968
              precision    recall  f1-score   support

           0       0.55      0.61      0.58       133
           1       0.49      0.43      0.45       115

    accuracy                           0.52       248
   macro avg       0.52      0.52      0.52       248
weighted avg       0.52      0.52      0.52       248



In [9]:
xgb_prob = xgb.predict_proba(X_test_scaled)[:, 1]
print("XGB ROC-AUC:", roc_auc_score(y_test, xgb_prob))

XGB ROC-AUC: 0.5599869238313174


In [10]:

importance = pd.Series(xgb.feature_importances_, index=feature_cols)
print(importance.sort_values(ascending=False))

dist_sma_20       0.102287
volatility_5d     0.100565
return_3d         0.090613
Volume            0.089769
macd_signal       0.089230
dist_sma_10       0.089096
volatility_10d    0.089025
return_5d         0.088293
rsi_14            0.087618
bb_width          0.087422
macd              0.086082
dtype: float32


In [11]:

prob = xgb.predict_proba(X_test_scaled[[-1]])[:, 1][0]

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print("Signal:", signal)
print("Confidence:", round(prob, 2))

Signal: BUY
Confidence: 0.78
